# Кейс 3. Шаг 2–3: витрина пользователей и гипотеза H3

**H3.** Как долго пользователи остаются на бесплатном тарифе до первой покупки, всегда ли покупке предшествует триал и сколько клиентов покупает без триала.

Ноутбук работает с Parquet на Drive (собран `case3_00_load_recon`). Результат: витрина `mart_users.parquet` (нужна следующим гипотезам), графики в `case3/fig/` и сводка `h3_results.txt` для отчёта.

**Приведение времени** (по итогам разведки, `case3_01_recon_extra`): поля без пояса в `users`, логах подключений, рекламе и прокси-логах записаны в московском времени, в `orders`, `transactions`, `subscribe` в UTC. Здесь всё переводится в UTC.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/case3"
import os, numpy as np, pandas as pd, duckdb, matplotlib.pyplot as plt
from google.colab import drive
drive.mount("/content/drive")
PARQUET, FIG = f"{DRIVE_DIR}/parquet", f"{DRIVE_DIR}/fig"
os.makedirs(FIG, exist_ok=True)
con = duckdb.connect()
con.execute("SET preserve_insertion_order = false")
for f in sorted(os.listdir(PARQUET)):
    if f.endswith(".parquet") and not f.startswith("user_connection_logs_"):
        con.execute(f"CREATE OR REPLACE VIEW {f[:-8]} AS SELECT * FROM '{PARQUET}/{f}'")

# Московское время → UTC для таблиц, где поле без пояса записано по Москве
con.execute("""CREATE OR REPLACE VIEW users_u AS SELECT * REPLACE (
    created_at - INTERVAL 3 HOUR AS created_at, updated_at - INTERVAL 3 HOUR AS updated_at,
    deleted_at - INTERVAL 3 HOUR AS deleted_at) FROM users""")
con.execute(f"""CREATE OR REPLACE VIEW conn_u AS SELECT * REPLACE (created_at - INTERVAL 3 HOUR AS created_at)
    FROM read_parquet('{PARQUET}/user_connection_logs_*.parquet')""")
con.execute("CREATE OR REPLACE VIEW adv_u AS SELECT * REPLACE (created_at - INTERVAL 3 HOUR AS created_at) FROM adv_view_history")

DATA_END = pd.Timestamp("2026-09-29 00:00")   # последние полные сутки в данных (UTC)
TRIAL_START = pd.Timestamp("2025-06-10")      # первый оплаченный триал в данных
CONN_START = pd.Timestamp("2026-02-01")       # начало логов подключений

REPORT = []
def out(*a):
    s = " ".join(str(x) for x in a); print(s); REPORT.append(s)
def save():
    open(f"{DRIVE_DIR}/h3_results.txt", "w", encoding="utf-8").write("\n".join(REPORT))

# Оформление графиков: светлая палитра, тонкие линии, приглушённая сетка
C = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "yellow": "#eda100", "violet": "#6250d6",
     "ink": "#0b0b0b", "ink2": "#52514e", "grid": "#e4e3df"}
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 200, "axes.edgecolor": C["ink2"], "axes.labelcolor": C["ink"],
    "xtick.color": C["ink2"], "ytick.color": C["ink2"], "axes.grid": True, "grid.color": C["grid"], "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10, "axes.titlesize": 11, "lines.linewidth": 2})

## 1. Витрина пользователей

Одна строка на пользователя. Платформа: самая частая ОС среди устройств пользователя (веб и расширения браузера отдельно). Заказы: первый оплаченный триал, первая оплаченная обычная покупка и её параметры. Подключения и реклама: агрегаты за весь период логов.

In [ ]:
con.execute(f"""
CREATE OR REPLACE TABLE mart AS
WITH dev AS (
    SELECT user_id,
           mode(CASE WHEN client_type IN ('web', 'web-extension') THEN 'web'
                     WHEN lower(os) = 'android' THEN 'android'
                     WHEN lower(os) = 'androidtv' THEN 'android_tv'
                     WHEN lower(os) = 'ios' THEN 'ios'
                     WHEN lower(os) LIKE 'windows%' THEN 'windows'
                     WHEN lower(os) LIKE 'mac%' THEN 'macos'
                     WHEN lower(os) IN ('linux', 'ubuntu', 'chromium os') THEN 'linux' END) AS platform,
           bool_or(lower(brand) IN ('huawei', 'honor')) AS huawei_honor,
           count(*) AS n_devices
    FROM devices WHERE user_id IS NOT NULL GROUP BY 1),
o AS (
    SELECT user_id,
           min(created_at) FILTER (WHERE trial = 't') AS first_trial_attempt,
           min(created_at) FILTER (WHERE trial = 't' AND status = 'paid') AS first_trial_paid,
           min(created_at) FILTER (WHERE trial = 'f') AS first_regular_attempt,
           min(created_at) FILTER (WHERE trial = 'f' AND status = 'paid') AS first_regular_paid,
           arg_min(product_id, created_at) FILTER (WHERE trial = 'f' AND status = 'paid') AS first_paid_product,
           arg_min(period_id, created_at) FILTER (WHERE trial = 'f' AND status = 'paid') AS first_paid_period,
           arg_min(pre_payment, created_at) FILTER (WHERE trial = 'f' AND status = 'paid') AS first_paid_pre_payment,
           count(*) FILTER (WHERE trial = 'f' AND status = 'paid') AS n_regular_paid,
           count(*) FILTER (WHERE trial = 't' AND status = 'failed') AS n_trial_failed,
           count(*) FILTER (WHERE trial = 'f' AND status = 'failed') AS n_regular_failed,
           sum(try_cast(amount AS DOUBLE)) FILTER (WHERE trial = 'f' AND status = 'paid' AND provider_id = '2') AS revenue_rub_yookassa
    FROM orders GROUP BY 1),
a AS (
    SELECT user_id, count(*) AS n_adv, count(DISTINCT CAST(created_at AS DATE)) AS adv_days,
           min(created_at) AS first_adv, max(created_at) AS last_adv
    FROM adv_u GROUP BY 1),
c AS (
    SELECT user_id, min(created_at) AS first_conn, max(created_at) AS last_conn,
           count(DISTINCT CAST(created_at AS DATE)) AS conn_days,
           count(DISTINCT CAST(created_at AS DATE)) FILTER (WHERE subscribe_id IS NULL) AS free_conn_days,
           count(*) FILTER (WHERE subscribe_id IS NULL) AS free_conn_n
    FROM conn_u GROUP BY 1)
SELECT u.id AS user_id, u.role, u.is_was_guest = 't' AS was_guest, u.deleted_at IS NOT NULL AS deleted,
       u.created_at AS reg_at, u.trial_used_at, strftime(u.created_at, '%Y-%m') AS reg_month,
       coalesce(dev.platform, 'unknown') AS platform, coalesce(dev.huawei_honor, false) AS huawei_honor,
       coalesce(dev.n_devices, 0) AS n_devices,
       o.* EXCLUDE (user_id), a.* EXCLUDE (user_id), c.* EXCLUDE (user_id)
FROM users_u u
LEFT JOIN dev ON dev.user_id = u.id
LEFT JOIN o ON o.user_id = u.id
LEFT JOIN a ON a.user_id = u.id
LEFT JOIN c ON c.user_id = u.id
""")
con.execute(f"COPY mart TO '{PARQUET}/mart_users.parquet' (FORMAT parquet, COMPRESSION zstd)")
out("# Витрина:", con.sql("SELECT count(*) FROM mart").fetchone()[0], "пользователей")
print(con.sql("SELECT role, platform, count(*) n FROM mart GROUP BY ALL ORDER BY n DESC LIMIT 20").df().to_string(index=False))

## 2. Пути к первой покупке

Под «покупкой» понимается первая **оплаченная обычная (не триальная) подписка**. Триал платный (10–30 ₽ за 3 дня тарифа Lite) и сам по себе покупкой подписки не считается.

Пути:
- **триал → покупка**: оплаченный триал раньше первой покупки;
- **покупка без триала**: оплаченного триала до первой покупки не было;
- **только триал**: триал оплачен, обычной покупки нет;
- **не платил**.

Гости в выборку не входят: заказов у них нет вообще (платить может только зарегистрированный пользователь). Берём клиентов, зарегистрированных с 10.06.2025 (когда появился триал).

In [ ]:
M = con.sql(f"""
    SELECT * FROM mart WHERE role = 'client' AND reg_at >= TIMESTAMP '{TRIAL_START}' AND reg_at < TIMESTAMP '{DATA_END}'
""").df()
for col in ["reg_at", "first_trial_paid", "first_regular_paid", "first_trial_attempt", "first_conn", "trial_used_at"]:
    M[col] = pd.to_datetime(M[col])

paid, trial = M["first_regular_paid"].notna(), M["first_trial_paid"].notna()
M["path"] = np.select(
    [paid & trial & (M["first_trial_paid"] < M["first_regular_paid"]), paid, trial],
    ["триал → покупка", "покупка без триала", "только триал"], default="не платил")

out("\n# 2. Пути клиентов (регистрация 10.06.2025–28.09.2026)")
out(f"клиентов: {len(M)}")
tab = M["path"].value_counts().rename("n").to_frame()
tab["share_all"] = (tab["n"] / len(M)).round(4)
payers = M[M["first_regular_paid"].notna()]
tab["share_payers"] = (tab["n"] / len(payers)).round(4).where(tab.index.isin(["триал → покупка", "покупка без триала"]))
out(tab.to_string())

anom = payers[payers["first_regular_paid"] < payers["reg_at"]]
out(f"покупка раньше регистрации (ошибки времени): {len(anom)}")

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (round(float(c - h), 4), round(float(c + h), 4))

k = (payers["path"] == "покупка без триала").sum()
out(f"доля покупок без триала: {k / len(payers):.3f}, 95% ДИ {wilson(k, len(payers))}")

out("\nпо платформам (только платящие):")
pt = pd.crosstab(payers["platform"], payers["path"], margins=True)
pt["share_no_trial"] = (pt.get("покупка без триала", 0) / pt["All"]).round(3)
out(pt.sort_values("All", ascending=False).to_string())

out("\nпо месяцу регистрации (только платящие):")
mt = pd.crosstab(payers["reg_month"], payers["path"])
mt["share_no_trial"] = (mt["покупка без триала"] / mt.sum(axis=1)).round(3)
out(mt.to_string())
save()

## 3. Триал: конверсия и момент покупки

Конверсия триала считается только для триалов, у которых после оплаты прошло не меньше 30 дней (иначе недавние триалы занижают долю). Отдельно смотрим, через сколько после триала происходит покупка: пик около 3 дней означает, что покупка совпадает с окончанием триала (вероятно, автосписание).

In [ ]:
T = M[M["first_trial_paid"].notna()].copy()
T["days_to_paid"] = (T["first_regular_paid"] - T["first_trial_paid"]).dt.total_seconds() / 86400
mature = T[T["first_trial_paid"] <= DATA_END - pd.Timedelta(days=30)]
conv30 = (mature["days_to_paid"].between(0, 30)).sum()
out("\n# 3. Триал")
out(f"оплаченных триалов (первых у клиента): {len(T)}; со «зрелым» окном 30 дней: {len(mature)}")
out(f"конверсия триала в покупку за 30 дней: {conv30 / len(mature):.3f}, 95% ДИ {wilson(conv30, len(mature))}")
out(f"конверсия за всё время (у зрелых): {mature['days_to_paid'].ge(0).sum() / len(mature):.3f}")

bins = [-0.001, 2.5, 3.5, 7, 30, np.inf]
labels = ["до 2,5 дн.", "2,5–3,5 дн. (конец триала)", "3,5–7 дн.", "7–30 дн.", "> 30 дн."]
conv = T[T["days_to_paid"] >= 0]
dist = pd.cut(conv["days_to_paid"], bins=bins, labels=labels).value_counts(sort=False)
out("через сколько после триала первая покупка (купившие):")
out((dist / dist.sum()).round(3).to_string())
out(f"медиана: {conv['days_to_paid'].median():.2f} дн.")

out("\nконверсия триала за 30 дней по месяцу триала:")
mature = mature.assign(m=mature["first_trial_paid"].dt.strftime("%Y-%m"), ok=mature["days_to_paid"].between(0, 30))
cm = mature.groupby("m")["ok"].agg(["sum", "count"])
cm["conv30"] = (cm["sum"] / cm["count"]).round(3)
out(cm.to_string())

out("\nконверсия триала за 30 дней по платформе:")
cp = mature.groupby("platform")["ok"].agg(["sum", "count"])
cp["conv30"] = (cp["sum"] / cp["count"]).round(3)
cp["ci95"] = [wilson(s, n) for s, n in zip(cp["sum"], cp["count"])]
out(cp.sort_values("count", ascending=False).to_string())

out(f"\nнеоплаченные попытки триала: у {(M['first_trial_attempt'].notna() & M['first_trial_paid'].isna()).sum()} клиентов "
    f"триал так и не прошёл оплату")
save()

## 4. Время от регистрации до первой покупки

Недавние пользователи ещё не успели купить: если просто взять медиану по купившим, оценка будет занижена (цензурирование). Поэтому строим кривую Каплана–Мейера: доля купивших к дню *t* после регистрации с учётом того, что у части пользователей наблюдение обрывается на конце данных.

In [ ]:
def km_cdf(durations, events, grid):
    """Оценка Каплана–Мейера: доля пользователей с событием к моменту t, то есть 1 − S(t)."""
    d = np.asarray(durations, float); e = np.asarray(events, bool)
    ev = pd.Series(1, index=d[e]).groupby(level=0).sum()          # число событий в каждый момент
    times = ev.index.values
    at_risk = len(d) - np.searchsorted(np.sort(d), times, side="left")
    S = np.cumprod(1 - ev.values / at_risk)
    idx = np.searchsorted(times, np.asarray(grid, float), side="right") - 1
    return np.where(idx >= 0, 1 - S[np.clip(idx, 0, None)], 0.0)

M["event"] = M["first_regular_paid"].notna()
M["t_end"] = M["first_regular_paid"].where(M["event"], DATA_END)
M["dur_days"] = ((M["t_end"] - M["reg_at"]).dt.total_seconds() / 86400).clip(lower=0)
GRID = [1, 3, 7, 14, 30, 60, 90, 180]

out("\n# 4. Доля купивших к дню t после регистрации (Каплан–Мейер)")
rows = {"все клиенты": km_cdf(M["dur_days"], M["event"], GRID)}
for q, g in M.groupby(pd.to_datetime(M["reg_at"]).dt.to_period("Q").astype(str)):
    if len(g) > 2000:
        rows[f"рег. {q}"] = km_cdf(g["dur_days"], g["event"], GRID)
for p in ["android", "ios", "windows", "macos", "web"]:
    g = M[M["platform"] == p]
    if len(g) > 500:
        rows[p] = km_cdf(g["dur_days"], g["event"], GRID)
km_tab = pd.DataFrame(rows, index=[f"{g} дн." for g in GRID]).T.round(4)
out(km_tab.to_string())

out("\nсреди купивших: время от регистрации до первой покупки, дни (квантили 25/50/75/90)")
for pth, g in payers.assign(d=(payers["first_regular_paid"] - payers["reg_at"]).dt.total_seconds() / 86400).groupby("path"):
    out(f"   {pth}: n={len(g)}, {np.round(g['d'].quantile([.25, .5, .75, .9]).values, 2).tolist()}, "
        f"в первые сутки: {(g['d'] < 1).mean():.3f}")
save()

## 5. Бесплатное использование до покупки

Логи подключений есть только с 01.02.2026, поэтому берём купивших клиентов, зарегистрированных не раньше этой даты: для них видна вся история бесплатных подключений и просмотров рекламы до покупки.

In [ ]:
con.register("payers_df", payers[["user_id", "reg_at", "first_regular_paid", "path", "platform"]])
F = con.sql(f"""
    SELECT p.user_id, p.path, p.platform,
           count(DISTINCT CAST(c.created_at AS DATE)) FILTER (WHERE c.subscribe_id IS NULL AND c.created_at < p.first_regular_paid) AS free_days_before,
           count(*) FILTER (WHERE c.subscribe_id IS NULL AND c.created_at < p.first_regular_paid) AS free_conn_before
    FROM payers_df p LEFT JOIN conn_u c ON c.user_id = p.user_id
    WHERE p.reg_at >= TIMESTAMP '{CONN_START}'
    GROUP BY 1, 2, 3""").df()
A = con.sql(f"""
    SELECT p.user_id, count(a.user_id) AS adv_before
    FROM payers_df p LEFT JOIN adv_u a ON a.user_id = p.user_id AND a.created_at < p.first_regular_paid
    WHERE p.reg_at >= TIMESTAMP '{CONN_START}' GROUP BY 1""").df()
F = F.merge(A, on="user_id", how="left")
bins = [-1, 0, 1, 3, 7, 30, np.inf]
labels = ["0 дней", "1 день", "2–3", "4–7", "8–30", "> 30"]
F["bucket"] = pd.cut(F["free_days_before"], bins=bins, labels=labels)
out(f"\n# 5. Бесплатные дни до покупки (купившие, рег. с 01.02.2026): n={len(F)}")
fb = pd.crosstab(F["bucket"], F["path"], normalize="columns").round(3)
fb["все"] = F["bucket"].value_counts(normalize=True, sort=False).round(3)
out(fb.to_string())
out("квантили бесплатных дней 50/75/90:", F["free_days_before"].quantile([.5, .75, .9]).tolist())
out(f"смотрели рекламу до покупки: {(F['adv_before'] > 0).mean():.3f}; медиана просмотров у смотревших: "
    f"{F.loc[F['adv_before'] > 0, 'adv_before'].median()}")
save()

## 6. Графики для отчёта

In [ ]:
# 6.1 Пути к первой покупке (доли среди купивших); цвет закреплён за путём
PATH_COLOR = {"триал → покупка": C["blue"], "покупка без триала": C["orange"]}
order = ["покупка без триала", "триал → покупка"]          # снизу вверх
pp = payers["path"].value_counts(normalize=True).reindex(order).fillna(0)
fig, ax = plt.subplots(figsize=(6.4, 1.9))
ax.barh(order, pp.values, color=[PATH_COLOR[k] for k in order], height=0.55)
for y, v in enumerate(pp.values):
    ax.text(v + 0.01, y, f"{v:.0%}", va="center", color=C["ink"])
ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
n_txt = f"{len(payers):,}".replace(",", "\u00a0")
ax.set_title(f"Путь к первой покупке, клиенты с 10.06.2025 (n = {n_txt})", loc="left")
ax.grid(axis="y", visible=False)
fig.tight_layout(); fig.savefig(f"{FIG}/h3_paths.png"); plt.show()

# 6.2 Через сколько дней после триала первая покупка
fig, ax = plt.subplots(figsize=(6.4, 3))
d = conv["days_to_paid"].clip(upper=30)
ax.hist(d, bins=np.arange(0, 30.5, 0.5), color=C["blue"], edgecolor="white", linewidth=0.6)
ax.axvline(3, color=C["ink2"], linewidth=1, linestyle="--")
ax.annotate("конец триала (3 дня)", xy=(3, ax.get_ylim()[1] * 0.85), xytext=(6, ax.get_ylim()[1] * 0.85),
            color=C["ink2"], va="center", arrowprops=dict(arrowstyle="-", color=C["ink2"], linewidth=0.8))
ax.set_xlabel("дней от оплаты триала до покупки (> 30 в последнем столбце)"); ax.set_ylabel("клиентов")
ax.set_title("Покупка после триала", loc="left")
fig.tight_layout(); fig.savefig(f"{FIG}/h3_trial_to_paid.png"); plt.show()

# 6.3 Доля купивших к дню t (Каплан–Мейер), по кварталам регистрации; подписи у концов линий
fig, ax = plt.subplots(figsize=(6.4, 3.4))
tgrid = np.arange(0, 91)
cohorts = [k for k in rows if k.startswith("рег.")][-4:]
colors = [C["blue"], C["orange"], C["aqua"], C["violet"]]
for col, q in zip(colors, cohorts):
    g = M[pd.to_datetime(M["reg_at"]).dt.to_period("Q").astype(str) == q.replace("рег. ", "")]
    y = km_cdf(g["dur_days"], g["event"], tgrid)
    ax.plot(tgrid, y, color=col, label=q.replace("рег. ", ""))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlabel("дней после регистрации"); ax.set_ylabel("доля купивших")
ax.set_title("Доля клиентов, купивших подписку (Каплан–Мейер)", loc="left")
ax.legend(frameon=False, loc="lower right", title="квартал регистрации", title_fontsize=9); ax.set_xlim(0, 90)
fig.tight_layout(); fig.savefig(f"{FIG}/h3_km.png"); plt.show()
out(f"\nГрафики: {FIG}/h3_paths.png, h3_trial_to_paid.png, h3_km.png")
save()